# Sideslip Prediction with Advanced Loss Functions

## Overview
This notebook implements an **RNN-based prediction model** for the Kaggle Drift Challenge to predict vehicle sideslip angle.

## Objective
Precise prediction of **sideSlip** (slip angle) based on time series features from vehicle data.

## Methodology

### 🏗️ Architecture
- **Model**: Recurrent Neural Network (RNN) with ReLU activation
- **Input**: Sequences of 20 timesteps with vehicle data (velocity, acceleration, etc.)
- **Output**: Single sideSlip value

### 🎯 Loss Functions (Modular)
1. **Standard L1 Loss**: Baseline Mean Absolute Error
2. **Weighted L1 Loss**: Higher weights for Few-Shot samples based on Kaggle Band Mapping
3. **Focal L1 Loss**: Focuses on difficult samples with large errors (Loss = |error|^gamma, gamma ≥ 1.0)

### 🔄 Two-Stage Training
- **Stage 1 (Warm-up)**: Training with Standard L1 Loss (30 Epochs, lr=0.0003)
- **Stage 2 (Fine-tuning)**: Training with Advanced Loss (40 Epochs, lr=0.00005)
  - Enables stable pre-training and targeted optimization for Few-Shot performance

### 🎲 Multi-Seed Ensembling
- Training multiple models with different seeds (e.g., [42, 2024])
- Averaging predictions for more robust results

### 🔀 Blending Strategy
- **Baseline Ensemble**: Multi-seed averaging of Standard L1 models
- **Advanced Ensemble**: Multi-seed averaging of Two-Stage models
- **Final Blend**: 50/50 mix of Baseline + Advanced for optimal generalization

## Features
- ✅ Configurable loss functions (L1, Weighted L1, Focal L1)
- ✅ Two-Stage training pipeline
- ✅ Multi-Seed ensembling
- ✅ Baseline + Advanced loss blending
- ✅ Automatic Kaggle submission file generation

## Setup & Imports

In [13]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

EXPECTED_ROWS = 49328

Device: cpu


## Shot Band Mapping

**Official Kaggle Mapping for Many/Medium/Few-Shot Bands**

This mapping categorizes sideSlip values by frequency in the training set:
- **Many-Shot**: [-5.9° to 3.5°] - Common values (many training examples)
- **Medium-Shot**: [-11.9° to -6.0°] and [3.6° to 8.1°] - Moderate number of samples
- **Few-Shot**: [-15.0° to -12.0°] and [8.2° to 12.0°] - Rare extreme values (few training examples)

**Usage**: Used by Weighted L1 Loss to assign higher weights to Few-Shot samples.

In [14]:
# Official Kaggle shot band mapping
shot_band_mapping = {
    -15.0: 'few', -14.9: 'few', -14.8: 'few', -14.7: 'few', -14.6: 'few',
    -14.5: 'few', -14.4: 'few', -14.3: 'few', -14.2: 'few', -14.1: 'few',
    -14.0: 'few', -13.9: 'few', -13.8: 'few', -13.7: 'few', -13.6: 'few',
    -13.5: 'few', -13.4: 'few', -13.3: 'few', -13.2: 'few', -13.1: 'few',
    -13.0: 'few', -12.9: 'few', -12.8: 'few', -12.7: 'few', -12.6: 'few',
    -12.5: 'few', -12.4: 'few', -12.3: 'few', -12.2: 'few', -12.1: 'few',
    -12.0: 'few', -11.9: 'medium', -11.8: 'medium', -11.7: 'medium',
    -11.6: 'medium', -11.5: 'medium', -11.4: 'medium', -11.3: 'medium',
    -11.2: 'medium', -11.1: 'medium', -11.0: 'medium', -10.9: 'medium',
    -10.8: 'medium', -10.7: 'medium', -10.6: 'medium', -10.5: 'medium',
    -10.4: 'medium', -10.3: 'medium', -10.2: 'medium', -10.1: 'medium',
    -10.0: 'medium', -9.9: 'medium', -9.8: 'medium', -9.7: 'medium',
    -9.6: 'medium', -9.5: 'medium', -9.4: 'medium', -9.3: 'medium',
    -9.2: 'medium', -9.1: 'medium', -9.0: 'medium', -8.9: 'medium',
    -8.8: 'medium', -8.7: 'medium', -8.6: 'medium', -8.5: 'medium',
    -8.4: 'medium', -8.3: 'medium', -8.2: 'medium', -8.1: 'medium',
    -8.0: 'medium', -7.9: 'medium', -7.8: 'medium', -7.7: 'medium',
    -7.6: 'medium', -7.5: 'medium', -7.4: 'medium', -7.3: 'medium',
    -7.2: 'medium', -7.1: 'medium', -7.0: 'medium', -6.9: 'medium',
    -6.8: 'medium', -6.7: 'medium', -6.6: 'medium', -6.5: 'medium',
    -6.4: 'medium', -6.3: 'medium', -6.2: 'medium', -6.1: 'medium',
    -6.0: 'medium', -5.9: 'many', -5.8: 'many', -5.7: 'many', -5.6: 'many',
    -5.5: 'many', -5.4: 'many', -5.3: 'many', -5.2: 'many', -5.1: 'many',
    -5.0: 'many', -4.9: 'many', -4.8: 'many', -4.7: 'many', -4.6: 'many',
    -4.5: 'many', -4.4: 'many', -4.3: 'many', -4.2: 'many', -4.1: 'many',
    -4.0: 'many', -3.9: 'many', -3.8: 'many', -3.7: 'many', -3.6: 'many',
    -3.5: 'many', -3.4: 'many', -3.3: 'many', -3.2: 'many', -3.1: 'many',
    -3.0: 'many', -2.9: 'many', -2.8: 'many', -2.7: 'many', -2.6: 'many',
    -2.5: 'many', -2.4: 'many', -2.3: 'many', -2.2: 'many', -2.1: 'many',
    -2.0: 'many', -1.9: 'many', -1.8: 'many', -1.7: 'many', -1.6: 'many',
    -1.5: 'many', -1.4: 'many', -1.3: 'many', -1.2: 'many', -1.1: 'many',
    -1.0: 'many', -0.9: 'many', -0.8: 'many', -0.7: 'many', -0.6: 'many',
    -0.5: 'many', -0.4: 'many', -0.3: 'many', -0.2: 'many', -0.1: 'many',
    0.0: 'many', 0.1: 'many', 0.2: 'many', 0.3: 'many', 0.4: 'many',
    0.5: 'many', 0.6: 'many', 0.7: 'many', 0.8: 'many', 0.9: 'many',
    1.0: 'many', 1.1: 'many', 1.2: 'many', 1.3: 'many', 1.4: 'many',
    1.5: 'many', 1.6: 'many', 1.7: 'many', 1.8: 'many', 1.9: 'many',
    2.0: 'many', 2.1: 'many', 2.2: 'many', 2.3: 'many', 2.4: 'many',
    2.5: 'many', 2.6: 'many', 2.7: 'many', 2.8: 'many', 2.9: 'many',
    3.0: 'many', 3.1: 'many', 3.2: 'many', 3.3: 'many', 3.4: 'many',
    3.5: 'many', 3.6: 'medium', 3.7: 'medium', 3.8: 'medium', 3.9: 'medium',
    4.0: 'medium', 4.1: 'medium', 4.2: 'medium', 4.3: 'medium',
    4.4: 'medium', 4.5: 'medium', 4.6: 'medium', 4.7: 'medium',
    4.8: 'medium', 4.9: 'medium', 5.0: 'medium', 5.1: 'medium',
    5.2: 'medium', 5.3: 'medium', 5.4: 'medium', 5.5: 'medium',
    5.6: 'medium', 5.7: 'medium', 5.8: 'medium', 5.9: 'medium',
    6.0: 'medium', 6.1: 'medium', 6.2: 'medium', 6.3: 'medium',
    6.4: 'medium', 6.5: 'medium', 6.6: 'medium', 6.7: 'medium',
    6.8: 'medium', 6.9: 'medium', 7.0: 'medium', 7.1: 'medium',
    7.2: 'medium', 7.3: 'medium', 7.4: 'medium', 7.5: 'medium',
    7.6: 'medium', 7.7: 'medium', 7.8: 'medium', 7.9: 'medium',
    8.0: 'medium', 8.1: 'medium', 8.2: 'few', 8.3: 'few', 8.4: 'few',
    8.5: 'few', 8.6: 'few', 8.7: 'few', 8.8: 'few', 8.9: 'few', 9.0: 'few',
    9.1: 'few', 9.2: 'few', 9.3: 'few', 9.4: 'few', 9.5: 'few', 9.6: 'few',
    9.7: 'few', 9.8: 'few', 9.9: 'few', 10.0: 'few', 10.1: 'few',
    10.2: 'few', 10.3: 'few', 10.4: 'few', 10.5: 'few', 10.6: 'few',
    10.7: 'few', 10.8: 'few', 10.9: 'few', 11.0: 'few', 11.1: 'few',
    11.2: 'few', 11.3: 'few', 11.4: 'few', 11.5: 'few', 11.6: 'few',
    11.7: 'few', 11.8: 'few', 11.9: 'few', 12.0: 'few'
}

print(f"Shot band mapping loaded: {len(shot_band_mapping)} values")

Shot band mapping loaded: 271 values


## Model Architecture

**Simple RNN with optimized hyperparameters**

### Architecture
- **RNN Layer**: 64 Hidden Units, ReLU Activation, 1 Layer
- **Fully Connected**: 64 Units with ReLU Activation
- **Output Layer**: Linear (1 output for sideSlip prediction)

### Optional: Residual Connections
- Can be activated with `use_residual=True`
- Skip-Connection: Adds RNN output directly to FC output
- Helps with gradient flow in deeper networks

**Design Rationale**: Simple architecture, as complexity primarily lies in loss functions and ensembling.

In [15]:
class RNN(nn.Module):
    def __init__(self, input_size, use_residual=False):
        super().__init__()
        self.use_residual = use_residual
        self.rnn = nn.RNN(input_size, 64, 1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(64, 64)
        self.out = nn.Linear(64, 1)
        self.act = nn.ReLU()
        
        # Residual mapping (wenn aktiviert)
        if self.use_residual:
            self.residual_map = nn.Identity()  # 64 -> 64, kein Mapping nötig
    
    def forward(self, x):
        x, _ = self.rnn(x)
        x_last = x[:, -1, :]  # Letzter Zeitschritt
        
        if self.use_residual:
            x_fc = self.act(self.fc(x_last))
            x_res = self.residual_map(x_last)
            x_combined = x_fc + x_res
            return self.out(x_combined)
        else:
            return self.out(self.act(self.fc(x_last)))

print("✅ RNN Model defined")

✅ RNN Model defined


## Loss Functions

**Four modular loss functions for different training strategies**

### 1. Standard L1 Loss
```
Loss = mean(|pred - target|)
```
- **Usage**: Baseline, Stage 1 training
- **Advantage**: Stable, well-researched, robust convergence

### 2. Weighted L1 Loss
```
Loss = mean(weight * |pred - target|)
weight = f(target, shot_band_mapping)
```
- **Usage**: Advanced training for Few-Shot performance
- **Parameters**:
  - `many_weight`: Weight for common values (e.g., 1.0)
  - `medium_weight`: Weight for moderate values (e.g., 2.0)
  - `few_weight`: Weight for rare values (e.g., 3.0)
- **Advantage**: Direct control over Few-Shot weighting

### 3. Focal L1 Loss
```
Loss = mean(|pred - target|^gamma)
```
- **Usage**: Focusing on difficult samples (large errors)
- **Parameters**:
  - `gamma`: Exponent for error amplification (**IMPORTANT: gamma ≥ 1.0**)
    - gamma = 1.0 → Standard L1 Loss
    - gamma > 1.0 → Large errors are penalized disproportionately
    - gamma < 1.0 → **UNSTABLE!** Leads to exploding gradients and NaN
  - `lambda_reg`: Optional L2 regularization on predictions
- **⚠️ Note**: Focuses on **hard samples** (large errors), NOT on **few-shot bands**

### 4. Weighted Focal L1 Loss ⭐ (NEW)
```
Loss = mean(weight * |pred - target|^gamma)
weight = f(target, shot_band_mapping)
```
- **Usage**: **Best for Few-Shot learning** - combines band weighting with focal emphasis
- **Parameters**:
  - `many_weight`, `medium_weight`, `few_weight`: Band weights (e.g., 1.0, 2.0, 4.0)
  - `gamma`: Error amplification exponent (≥ 1.0, e.g., 1.5)
- **Advantage**: 
  - Focuses on **few-shot target bands** (via weights)
  - Emphasizes **difficult samples** within those bands (via gamma)
  - Combines best of both worlds

### Loss Function Factory
`get_loss_function(loss_type, **kwargs)` enables easy switching between loss types.

**Recommendation for Few-Shot**: Use `'weighted_focal_l1'` or `'weighted_l1'`

In [16]:
class WeightedL1Loss(nn.Module):
    """
    Weighted L1 Loss using official Kaggle band mapping.
    Assigns higher weights to few-shot samples.
    """
    
    def __init__(self, many_weight=1.0, medium_weight=2.0, few_weight=3.0):
        super().__init__()
        self.many_weight = many_weight
        self.medium_weight = medium_weight
        self.few_weight = few_weight
        
        # Pre-compute weight lookup table for efficiency
        sideslip_values = torch.arange(-150, 121, dtype=torch.float32) / 10.0
        self.weight_lookup = torch.ones(len(sideslip_values), dtype=torch.float32)
        
        for i, val in enumerate(sideslip_values):
            band = shot_band_mapping.get(val.item(), 'many')
            if band == 'few':
                self.weight_lookup[i] = self.few_weight
            elif band == 'medium':
                self.weight_lookup[i] = self.medium_weight
            else:
                self.weight_lookup[i] = self.many_weight
        
        self.min_val = -15.0
        self.max_val = 12.0
    
    def forward(self, pred, target):
        # Round to match band mapping
        rounded_target = torch.round(target * 10) / 10
        
        # Clamp and convert to indices
        clamped = torch.clamp(rounded_target, self.min_val, self.max_val)
        indices = ((clamped - self.min_val) * 10).long().cpu()
        
        # Vectorized lookup
        weights = self.weight_lookup[indices].to(target.device).reshape(target.shape)
        
        # Weighted L1 loss
        loss = weights * torch.abs(pred - target)
        return loss.mean()
    
    def __repr__(self):
        return f"WeightedL1Loss(many={self.many_weight}, medium={self.medium_weight}, few={self.few_weight})"


class FocalL1Loss(nn.Module):
    """
    Focal L1 Loss - focuses on difficult samples with large errors.
    Loss = (|pred - target|^gamma) where gamma > 1 emphasizes larger errors.
    
    Note: This does NOT focus on few-shot bands, only on hard-to-predict samples.
    Use WeightedFocalL1Loss for few-shot optimization.
    """
    
    def __init__(self, gamma=1.5, lambda_reg=0.0):
        super().__init__()
        self.gamma = gamma
        self.lambda_reg = lambda_reg
    
    def forward(self, pred, target):
        abs_error = torch.abs(pred - target)
        focal_loss = torch.pow(abs_error, self.gamma)
        
        # Optional L2 regularization on predictions
        if self.lambda_reg > 0:
            reg_term = self.lambda_reg * torch.mean(pred ** 2)
            return focal_loss.mean() + reg_term
        
        return focal_loss.mean()
    
    def __repr__(self):
        return f"FocalL1Loss(gamma={self.gamma}, lambda_reg={self.lambda_reg})"


class WeightedFocalL1Loss(nn.Module):
    """
    Weighted Focal L1 Loss - combines band weighting with focal emphasis.
    Loss = weight(target_band) * |pred - target|^gamma
    
    This focuses on:
    1. Few-shot samples (via band weights)
    2. Large errors within those samples (via gamma)
    
    Best for Few-Shot learning with emphasis on difficult cases.
    """
    
    def __init__(self, many_weight=1.0, medium_weight=2.0, few_weight=3.0, gamma=1.5):
        super().__init__()
        self.many_weight = many_weight
        self.medium_weight = medium_weight
        self.few_weight = few_weight
        self.gamma = gamma
        
        # Pre-compute weight lookup table for efficiency
        sideslip_values = torch.arange(-150, 121, dtype=torch.float32) / 10.0
        self.weight_lookup = torch.ones(len(sideslip_values), dtype=torch.float32)
        
        for i, val in enumerate(sideslip_values):
            band = shot_band_mapping.get(val.item(), 'many')
            if band == 'few':
                self.weight_lookup[i] = self.few_weight
            elif band == 'medium':
                self.weight_lookup[i] = self.medium_weight
            else:
                self.weight_lookup[i] = self.many_weight
        
        self.min_val = -15.0
        self.max_val = 12.0
    
    def forward(self, pred, target):
        # Round to match band mapping
        rounded_target = torch.round(target * 10) / 10
        
        # Clamp and convert to indices
        clamped = torch.clamp(rounded_target, self.min_val, self.max_val)
        indices = ((clamped - self.min_val) * 10).long().cpu()
        
        # Vectorized lookup
        weights = self.weight_lookup[indices].to(target.device).reshape(target.shape)
        
        # Weighted Focal L1 loss: weight * |error|^gamma
        abs_error = torch.abs(pred - target)
        focal_error = torch.pow(abs_error, self.gamma)
        loss = weights * focal_error
        return loss.mean()
    
    def __repr__(self):
        return f"WeightedFocalL1Loss(many={self.many_weight}, medium={self.medium_weight}, few={self.few_weight}, gamma={self.gamma})"


def get_loss_function(loss_type='l1', **kwargs):
    """
    Factory function to create loss functions.
    
    Args:
        loss_type: 'l1', 'weighted_l1', 'focal_l1', or 'weighted_focal_l1'
        **kwargs: Parameters for the specific loss function
    
    Returns:
        Loss function instance
    """
    if loss_type == 'l1':
        return nn.L1Loss()
    
    elif loss_type == 'weighted_l1':
        many_weight = kwargs.get('many_weight', 1.0)
        medium_weight = kwargs.get('medium_weight', 2.0)
        few_weight = kwargs.get('few_weight', 3.0)
        return WeightedL1Loss(many_weight, medium_weight, few_weight)
    
    elif loss_type == 'focal_l1':
        gamma = kwargs.get('gamma', 1.5)
        lambda_reg = kwargs.get('lambda_reg', 0.0)
        return FocalL1Loss(gamma, lambda_reg)
    
    elif loss_type == 'weighted_focal_l1':
        many_weight = kwargs.get('many_weight', 1.0)
        medium_weight = kwargs.get('medium_weight', 2.0)
        few_weight = kwargs.get('few_weight', 3.0)
        gamma = kwargs.get('gamma', 1.5)
        return WeightedFocalL1Loss(many_weight, medium_weight, few_weight, gamma)
    
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")

print("✅ Loss functions defined")
print("   Available: L1, Weighted L1, Focal L1, Weighted Focal L1")

✅ Loss functions defined
   Available: L1, Weighted L1, Focal L1, Weighted Focal L1


## Utility Functions

**Helper functions for prediction and submission**

In [17]:
def predict(model, loader):
    """Generate predictions from a trained model."""
    model.eval()
    preds = []
    with torch.no_grad():
        for x, _ in loader:
            preds.append(model(x.to(device)).cpu().numpy())
    return np.concatenate(preds).flatten()


def save_submission(preds, filename):
    """Save predictions to CSV file."""
    assert len(preds) == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS}, got {len(preds)}"
    df = pd.DataFrame({"id": range(len(preds)), "predicted_sideslip": preds})
    df.to_csv(filename, index=False)
    print(f"Saved: {filename}")
    print(f"  Mean: {preds.mean():.4f}, Std: {preds.std():.4f}")

print("✅ Utility functions ready")

✅ Utility functions ready


## Data Loading

Loading training and test data with preprocessing:
- **Train/Val Split**: 98% training, 2% validation
- **Sequence Length**: 20 timesteps
- **Threshold**: Only samples with vxCG ≥ 3.0 m/s (minimum velocity)
- **Dropped Features**: sideSlip (target), vyCG (leakage), time, longitude, latitude
- **Scaling**: StandardScaler on features

In [18]:
print("Loading data...")
train_loader, val_loader, pipeline = load_data(
    train_path="../datasets_challenge/train",
    train_ratio=0.98,
    batchsize=64,
    columns_to_drop=["sideSlip", "vyCG", "time", "longitude", "latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42,
)

test_loader = load_test_data(
    "../datasets_challenge/test",
    512,
    pipeline,
    load_anonymized_data=True
)

input_size = next(iter(train_loader))[0].shape[2]
test_count = sum(x.shape[0] for x, _ in test_loader)

print(f"\n✅ Data loaded")
print(f"   Input features: {input_size}")
print(f"   Test samples: {test_count}")
assert test_count == EXPECTED_ROWS

Loading data...
../datasets_challenge/train/train_part_001.csv
../datasets_challenge/train/train_part_002.csv
../datasets_challenge/train/train_part_003.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_013.csv
../datasets_challenge/train/train_part_014.csv
../datasets_challenge/train/train_part_015.csv
../datasets_challenge/train/train_part_016.csv
../datasets_challenge/train/train_part_017.csv
../datasets_challenge/train/train_part_018.csv
../datasets_challenge/train/train_part_019.csv
../datasets_challenge/train/train_part_020.csv
../datasets_challenge/train/train_part_021.c

## Training Functions

**Two training modes: Two-Stage and Baseline**

### Two-Stage Training (`train_two_stage`)
Two-stage training for optimal Few-Shot performance:

**Stage 1 - Warm-up with L1 Loss**
- Epochs: Configurable (default: 30)
- Learning Rate: 0.0003
- Loss: Standard L1 (stable, fast convergence)
- Goal: Solid base weights for the model

**Stage 2 - Fine-tuning with Advanced Loss**
- Epochs: Configurable (default: 40)
- Learning Rate: Reduced (default: 0.00005)
- Loss: Weighted L1 or Focal L1 (Few-Shot optimized)
- Validation: With L1 Loss for fair comparison
- Goal: Targeted improvement on difficult samples

**Parameters**:
- `seed`: Random seed for reproducibility
- `s1_epochs`, `s2_epochs`: Number of epochs per stage
- `s2_lr`: Learning rate for Stage 2
- `loss_type`: 'weighted_l1' or 'focal_l1'
- `use_residual`: Enable residual connections
- `**loss_kwargs`: Parameters for the chosen loss function

### Baseline Training (`train_baseline`)
Single-stage training as comparison baseline:
- Epochs: Max 50 with Early Stopping (Patience: 10)
- Learning Rate: 0.0003
- Loss: Standard L1
- Goal: Robust baseline for ensembling

**Return**: Trained PyTorch model (state_dict from best validation checkpoint)

In [19]:
def train_two_stage(
    seed,
    s1_epochs=30,
    s2_epochs=40,
    s2_lr=0.00005,
    loss_type='weighted_l1',
    use_residual=False,
    **loss_kwargs
):
    """
    Two-stage training:
    - Stage 1: L1 Loss
    - Stage 2: Custom Loss Function
    
    Args:
        seed: Random seed
        s1_epochs: Stage 1 epochs
        s2_epochs: Stage 2 epochs
        s2_lr: Stage 2 learning rate
        loss_type: 'weighted_l1' or 'focal_l1'
        use_residual: Enable residual connections
        **loss_kwargs: Parameters for loss function
    """
    seed_everything(seed)
    model = RNN(input_size, use_residual=use_residual).to(device)
    
    # ========== Stage 1: L1 Loss ==========
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state = np.inf, None
    
    for epoch in range(s1_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            loss = crit(model(x.to(device)), y.to(device).reshape(-1, 1))
            loss.backward()
            opt.step()
        
        model.eval()
        val_loss = sum(
            crit(model(x.to(device)), y.to(device).reshape(-1, 1)).item()
            for x, y in val_loader
        ) / len(val_loader)
        
        if val_loss < best:
            best, state = val_loss, model.state_dict().copy()
    
    model.load_state_dict(state)
    print(f"  Stage 1 complete | Best Val Loss: {best:.4f}")
    
    # ========== Stage 2: Custom Loss ==========
    opt = torch.optim.Adam(model.parameters(), lr=s2_lr)
    crit2 = get_loss_function(loss_type, **loss_kwargs)
    best2, state2 = np.inf, None
    
    for epoch in range(s2_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            loss = crit2(model(x.to(device)), y.to(device).reshape(-1, 1))
            loss.backward()
            opt.step()
        
        # Validate with same loss as training
        model.eval()
        val_loss = sum(
            crit2(model(x.to(device)), y.to(device).reshape(-1, 1)).item()
            for x, y in val_loader
        ) / len(val_loader)
        
        if val_loss < best2:
            best2, state2 = val_loss, model.state_dict().copy()
    
    model.load_state_dict(state2)
    print(f"  Stage 2 complete | Best Val Loss: {best2:.4f} | Loss: {crit2}")
    
    return model


def train_baseline(seed, use_residual=False):
    """Baseline single-stage training with L1 Loss."""
    seed_everything(seed)
    model = RNN(input_size, use_residual=use_residual).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state, patience = np.inf, None, 0
    
    for epoch in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            loss = crit(model(x.to(device)), y.to(device).reshape(-1, 1))
            loss.backward()
            opt.step()
        
        model.eval()
        val_loss = sum(
            crit(model(x.to(device)), y.to(device).reshape(-1, 1)).item()
            for x, y in val_loader
        ) / len(val_loader)
        
        if val_loss < best:
            best, state, patience = val_loss, model.state_dict().copy(), 0
        else:
            patience += 1
            if patience > 10:
                break
    
    model.load_state_dict(state)
    print(f"  Baseline complete | Best Val Loss: {best:.4f}")
    
    return model

print("✅ Training functions ready")

✅ Training functions ready


## Configuration

**Central configuration for the experiment**

Modify parameters here for your experiment:
- Seeds for ensembling
- Model architecture (Residual connections)
- Training hyperparameters (Epochs, Learning rate)
- Loss function and its parameters
- Output filename

In [20]:
# ========== CONFIGURATION ==========

# Random Seeds for Ensemble
SEEDS = [42, 2024]

# Model Configuration
USE_RESIDUAL = False  # Enable residual connections?

# Two-Stage Training
S1_EPOCHS = 30        # Stage 1 Epochs (L1 Loss)
S2_EPOCHS = 40        # Stage 2 Epochs (Custom Loss)
S2_LR = 0.00005       # Stage 2 Learning Rate

# Loss Function Selection
LOSS_TYPE = 'weighted_focal_l1'  # Options: 'weighted_l1', 'focal_l1', 'weighted_focal_l1'

# Weighted L1 Parameters (if LOSS_TYPE='weighted_l1')
WEIGHTED_L1_CONFIG = {
    'many_weight': 1.0,
    'medium_weight': 2.0,
    'few_weight': 4.0,
}

# Focal L1 Parameters (if LOSS_TYPE='focal_l1')
# WARNING: This does NOT focus on few-shot bands, only on large errors!
FOCAL_L1_CONFIG = {
    'gamma': 1.5,  # Must be >= 1.0 for stable training
    'lambda_reg': 0.0,
}

# Weighted Focal L1 Parameters (if LOSS_TYPE='weighted_focal_l1')
# RECOMMENDED for Few-Shot learning
WEIGHTED_FOCAL_L1_CONFIG = {
    'many_weight': 1.0,
    'medium_weight': 2.0,
    'few_weight': 4.0,
    'gamma': 1.5,  # Must be >= 1.0 for stable training
}

# Output Filename
OUTPUT_FILE = "submission_weighted_focal_l1.csv"

print("="*80)
print("⚙️  CONFIGURATION")
print("="*80)
print(f"Seeds: {SEEDS}")
print(f"Residual Connections: {USE_RESIDUAL}")
print(f"Stage 1: {S1_EPOCHS} epochs, lr=0.0003, L1 Loss")
print(f"Stage 2: {S2_EPOCHS} epochs, lr={S2_LR}, {LOSS_TYPE.upper()}")
if LOSS_TYPE == 'weighted_l1':
    print(f"  Weights: {WEIGHTED_L1_CONFIG}")
elif LOSS_TYPE == 'focal_l1':
    print(f"  Params: {FOCAL_L1_CONFIG}")
    print(f"  ⚠️  WARNING: Focal L1 focuses on large errors, NOT few-shot bands!")
elif LOSS_TYPE == 'weighted_focal_l1':
    print(f"  Params: {WEIGHTED_FOCAL_L1_CONFIG}")
    print(f"  ✅ Few-Shot optimized with focal emphasis")
print(f"Output: {OUTPUT_FILE}")
print("="*80)

⚙️  CONFIGURATION
Seeds: [42, 2024]
Residual Connections: False
Stage 1: 30 epochs, lr=0.0003, L1 Loss
Stage 2: 40 epochs, lr=5e-05, WEIGHTED_FOCAL_L1
  Params: {'many_weight': 1.0, 'medium_weight': 2.0, 'few_weight': 4.0, 'gamma': 1.5}
  ✅ Few-Shot optimized with focal emphasis
Output: submission_weighted_focal_l1.csv


## Training Pipeline

**Complete training and ensemble pipeline**

### Workflow:
1. **Baseline Training**: Multi-Seed Standard L1 models
2. **Advanced Training**: Multi-Seed Two-Stage models with custom loss
3. **Ensembling**: Averaging within each group
4. **Blending**: 50/50 mix of Baseline + Advanced ensembles
5. **Submission**: Save final predictions as CSV

### Why Blending?
- **Baseline**: Robust on common values, generalizes well
- **Advanced**: Optimized for Few-Shot samples, specialized
- **Blend**: Combines stability + specialization for best overall performance

In [21]:
print("="*80)
print("🚀 TRAINING BASELINE MODELS")
print("="*80)

baseline_preds = []
for seed in SEEDS:
    print(f"\nTraining Baseline (seed={seed})")
    model = train_baseline(seed, use_residual=USE_RESIDUAL)
    preds = predict(model, test_loader)
    baseline_preds.append(preds)

baseline_ens = np.mean(baseline_preds, axis=0)
print(f"\n✅ Baseline Ensemble: mean={baseline_ens.mean():.4f}, std={baseline_ens.std():.4f}")

🚀 TRAINING BASELINE MODELS

Training Baseline (seed=42)
  Baseline complete | Best Val Loss: 0.1164

Training Baseline (seed=2024)
  Baseline complete | Best Val Loss: 0.1145

✅ Baseline Ensemble: mean=0.0076, std=1.5022


In [22]:
print("="*80)
print("🚀 TRAINING TWO-STAGE MODELS")
print("="*80)

# Select Loss Config based on LOSS_TYPE
if LOSS_TYPE == 'weighted_l1':
    loss_kwargs = WEIGHTED_L1_CONFIG
elif LOSS_TYPE == 'focal_l1':
    loss_kwargs = FOCAL_L1_CONFIG
elif LOSS_TYPE == 'weighted_focal_l1':
    loss_kwargs = WEIGHTED_FOCAL_L1_CONFIG
else:
    loss_kwargs = {}

advanced_preds = []
for seed in SEEDS:
    print(f"\nTraining Two-Stage (seed={seed})")
    model = train_two_stage(
        seed=seed,
        s1_epochs=S1_EPOCHS,
        s2_epochs=S2_EPOCHS,
        s2_lr=S2_LR,
        loss_type=LOSS_TYPE,
        use_residual=USE_RESIDUAL,
        **loss_kwargs
    )
    preds = predict(model, test_loader)
    advanced_preds.append(preds)

advanced_ens = np.mean(advanced_preds, axis=0)
print(f"\n✅ Advanced Ensemble: mean={advanced_ens.mean():.4f}, std={advanced_ens.std():.4f}")

🚀 TRAINING TWO-STAGE MODELS

Training Two-Stage (seed=42)
  Stage 1 complete | Best Val Loss: 0.1284
  Stage 2 complete | Best Val Loss: 0.0551 | Loss: WeightedFocalL1Loss(many=1.0, medium=2.0, few=4.0, gamma=1.5)

Training Two-Stage (seed=2024)
  Stage 1 complete | Best Val Loss: 0.1249
  Stage 2 complete | Best Val Loss: 0.0494 | Loss: WeightedFocalL1Loss(many=1.0, medium=2.0, few=4.0, gamma=1.5)

✅ Advanced Ensemble: mean=0.0095, std=1.4668


In [23]:
print("="*80)
print("🎯 FINAL BLENDING")
print("="*80)

# 50/50 Blend: Baseline + Advanced
final_blend = (baseline_ens + advanced_ens) / 2

print(f"\nBaseline Ensemble: mean={baseline_ens.mean():.4f}, std={baseline_ens.std():.4f}")
print(f"Advanced Ensemble: mean={advanced_ens.mean():.4f}, std={advanced_ens.std():.4f}")
print(f"Final Blend:       mean={final_blend.mean():.4f}, std={final_blend.std():.4f}")

save_submission(final_blend, OUTPUT_FILE)

🎯 FINAL BLENDING

Baseline Ensemble: mean=0.0076, std=1.5022
Advanced Ensemble: mean=0.0095, std=1.4668
Final Blend:       mean=0.0086, std=1.4839
Saved: submission_weighted_focal_l1.csv
  Mean: 0.0086, Std: 1.4839


## Experiments (Optional)

**Experimentation area for further testing**

Use this cell for:
- Grid search over loss parameters (e.g., different gamma values)
- Ablation studies (e.g., Baseline only vs Advanced only)
- Alternative blending weights
- Different seed combinations

Example code is available commented out.

In [24]:
# Beispiel: Experimente mit verschiedenen Loss-Konfigurationen

# # Experiment 1: Weighted L1 mit unterschiedlichen Gewichten
# configs = [
#     {'many_weight': 1.0, 'medium_weight': 2.0, 'few_weight': 3.0},
#     {'many_weight': 1.0, 'medium_weight': 2.0, 'few_weight': 5.0},
#     {'many_weight': 1.0, 'medium_weight': 1.5, 'few_weight': 4.0},
# ]

# for i, config in enumerate(configs):
#     print(f"\n{'='*80}")
#     print(f"Experiment {i+1}: {config}")
#     print(f"{'='*80}")
#     
#     preds = []
#     for seed in SEEDS:
#         model = train_two_stage(
#             seed=seed,
#             s1_epochs=30,
#             s2_epochs=40,
#             s2_lr=0.00005,
#             loss_type='weighted_l1',
#             use_residual=False,
#             **config
#         )
#         preds.append(predict(model, test_loader))
#     
#     ens = np.mean(preds, axis=0)
#     blend = (baseline_ens + ens) / 2
#     save_submission(blend, f"submission_exp{i+1}.csv")

print("Nutzen Sie diese Zelle für eigene Experimente!")

Nutzen Sie diese Zelle für eigene Experimente!
